# Phase 5 v7b: Doubled Orientation Carrier Test
No IO. One cell per claim.

In [ ]:
import numpy as np, math, cmath

def K(D, sign=-1):
    r=np.arange(D).reshape((D,1)); s=np.arange(D).reshape((1,D))
    return (1/math.sqrt(D))*np.exp(sign*2j*math.pi*r*s/D)

def G(D):
    r=np.arange(D); return np.exp(1j*math.pi*r*r/D)

def rev(D):
    R=np.zeros((D,D), dtype=complex)
    for r in range(D): R[r,(-r)%D]=1
    return R

def max_abs(A): return float(np.max(np.abs(A)))
print('SETUP PASS')

In [ ]:
# Claim 1: doubled carrier works for odd and even base moduli
threshold=1e-10
maxu=maxr=maxp=0.0
failed=[]
for n in range(1,61):
    D=2*n; Km=K(D,-1); g=G(D); I=np.eye(D); R=rev(D)
    u=max_abs(Km@Km.conj().T-I)
    r=max_abs(Km@Km-R)
    p=0.0
    for a in range(D):
        for b in range(D):
            p=max(p, abs(g[(a+b)%D]/(g[a]*g[b])-cmath.exp(2j*math.pi*a*b/D)))
    maxu=max(maxu,u); maxr=max(maxr,r); maxp=max(maxp,p)
    if max(u,r,p)>threshold: failed.append((n,u,r,p))
print('PASS' if not failed else 'FAIL', {'max_unitarity':maxu, 'max_reversal':maxr, 'max_polarization':maxp, 'failures':failed[:3]})

In [ ]:
# Claim 2: folded carrier fails exactly for odd carrier sizes
threshold=1e-10
odd_fail=True; even_pass=True
rows=[]
for N in range(1,31):
    g=np.array([cmath.exp(1j*math.pi*r*r/N) for r in range(N)])
    rep=max(abs(cmath.exp(1j*math.pi*(r+N)*(r+N)/N)-g[r]) for r in range(N))
    rows.append((N, rep))
    if N%2==1 and rep<threshold: odd_fail=False
    if N%2==0 and rep>=threshold: even_pass=False
print('PASS' if odd_fail and even_pass else 'FAIL', rows[:10])

In [ ]:
# Claim 3: old even carrier examples are recovered by base-modulus renaming
threshold=1e-12
failed=[]
for D in [8,10,12,26,60]:
    n=D//2
    err=max_abs(G(D)-G(2*n)) + max_abs(K(D,-1)-K(2*n,-1))
    if err>threshold: failed.append((D,n,err))
print('PASS' if not failed else 'FAIL', failed)